# Solutions — Components & composition

Only look here after you've actually tried the exercises in `components.ipynb`.

Every exercise in this topic happens in `playground/src/experiments/01-components.jsx`, so
these solutions are written out rather than run.

### LESSON 4 — Exercise

**1 and 2 — a component used twice.**

```jsx
function Footer() {
  return <p>Built by Sam</p>;
}

// …inside Experiment01, at the bottom:
<Footer />
<Footer />
```

**3 — two elements, no wrapper.** Vite refuses to build the file at all:

```text
[PARSE_ERROR] Adjacent JSX elements must be wrapped in an enclosing tag.
Help: Did you want a JSX fragment `<>...</>`?
```

The fix for now is a container element:

```jsx
function PageTitle() {
  return (
    <div>
      <h1>Dashboard</h1>
      <p>Everything at a glance</p>
    </div>
  );
}
```

The error's own suggestion — `<>...</>` — is a **fragment**, and it is the better answer.
LESSON 10 covers it. Using a `<div>` here is not wrong; it just adds an element to the page
that you did not really want.

### LESSON 4 — Mini challenge

**1. Renaming the definition only.** The page goes blank and the browser console shows:

```text
Uncaught ReferenceError: Greeting is not defined
```

Vite transforms the file happily — `<Greeting />` is perfectly valid syntax, and nothing
checks that `Greeting` exists until the code runs. So this one is a **runtime** failure, and
the browser console is where it appears.

Compare it with the adjacent-elements error. That one never compiled, so it is reported by
the dev server: in the terminal, **and** pushed to the page as Vite's full-screen error
overlay. Two different mechanisms, two different-looking failures — but do not conclude that
compile errors stay in the terminal. The overlay is usually the first thing you see.

**2. Renaming both to `greeting`.** Nothing crashes. The two greetings simply disappear, and
the console says:

```text
The tag <greeting> is unrecognized in this browser.
If you meant to render a React component, start its name with an uppercase letter.
```

Look in the page's elements panel and you will find `<greeting></greeting>` sitting there,
empty. React created an HTML element with that name and never called your function.

**3. Why React cannot just guess.** The decision is already made before React sees anything.
The build tool compiles the two forms differently:

```text
<Greeting />   becomes   jsx(Greeting, {})      the identifier — your function
<greeting />   becomes   jsx("greeting", {})    a string — an HTML tag name
```

By the time React receives it, one is a function reference and the other is the text
`"greeting"`. There is no function left to find. And it could not safely guess anyway:
lowercase names are legal HTML and legal custom elements, so treating them as components
would break real markup.

### LESSON 5 — Exercise

**1. A wrapper of your own.**

```jsx
function Panel({ children }) {
  return (
    <div>
      <h3>Panel</h3>
      {children}
    </div>
  );
}
```

**2. `<Panel />` with nothing inside.** The heading renders and nothing else. No error:
`children` is simply absent, and rendering nothing is perfectly legal in React.

**3. Nesting.** It works. A `<Card>` inside a `<Card>` gives you a `<section>` inside a
`<section>` — the outer card's `children` happens to contain another card, and it neither
knows nor cares. This is the whole point of composition.

**4. Deleting `{children}`.** The nested content vanishes from the page. No error, no
warning — the `<hr />`s still render, and the content you passed in is just never placed
anywhere.

That silence is worth remembering. When a component you wrote renders *something* but not
the content you gave it, a missing `{children}` is the first thing to check.

### LESSON 5 — Mini challenge

**1. Counting.** Design A needs **one** component. Design B needs **five**.

When the designer asks for a thicker border: in A you edit `Card` once and all five change.
In B you edit five files, and the bug is that you will edit four of them and miss one. The
fifth card looking slightly wrong is a genuinely common bug, and this is where it comes
from.

**2. When B is right.** When the thing really is one of a kind. A `SiteHeader`, a
`LoginPanel`, a `CheckoutSummary` — used once, with content that belongs to it and nowhere
else. Wrapping those in a generic `Card` and passing the content in from outside buys you
nothing and costs you a layer of indirection.

The test is not "is it repeated" but "does this component's content belong to it".

### LESSON 6 — Exercise

One reasonable split. Yours may differ — what matters is that the names are honest and
`StorePage` reads at a glance.

```jsx
function StoreHeader() {
  return (
    <div>
      <h1>Corner Store</h1>
      <p>Open 8:00 – 20:00</p>
    </div>
  );
}

function Section({ children }) {
  return (
    <div>
      <hr />
      {children}
    </div>
  );
}

function StoreFooter() {
  return (
    <div>
      <hr />
      <p>Corner Store, 2026</p>
    </div>
  );
}

function StorePage() {
  return (
    <div>
      <StoreHeader />
      <Section>
        <h2>Today</h2>
        <p>Fresh bread from 7:30</p>
        <p>Coffee beans restocked</p>
      </Section>
      <Section>
        <h2>Contact</h2>
        <p>hello@cornerstore.example</p>
        <p>+39 000 000 000</p>
      </Section>
      <StoreFooter />
    </div>
  );
}
```

**Why `Section` takes `children` and the others do not.** `Section` is used twice with
different contents, so its contents cannot live inside it. `StoreHeader` and `StoreFooter`
are used once and their content belongs to them — passing it in from outside would be
ceremony.

**The common mistake here** is extracting `Today` and `Contact` as two separate components
(`TodaySection`, `ContactSection`). It works, but you have written the `<hr />` wrapper
twice and gained nothing. Spotting that both share a shape, and that only the contents
differ, is the actual skill this exercise is training.

### LESSON 6 — Mini challenge

1. **`<Divider />` — leave it alone.** It saves you nothing: `<Divider />` and `<hr />` are
   the same length, and now a reader has to open another file to learn that it is an `<hr />`.
   Extract it later if dividers ever need to look different.

2. **`<PageWrapper>` — extract it.** Used once, and worth it: it has a clear name, a single
   job, and it keeps `App` readable.

3. **`<StatusBadge />` — not possible until topic 6.** Each row needs a *different* label.
   Right now a component either hardcodes its content or takes `children`, and neither gives
   you a badge that varies by row. Props are exactly this.

4. **`<SiteHeader />` — extract it.** Fifteen lines with an obvious name and one job. Used
   once is irrelevant.

**What separates 2 and 4 from 1.** All three are used once. The difference is not reuse at
all — it is whether the name earns its keep. `PageWrapper` and `SiteHeader` replace a block
you would otherwise have to read to identify. `Divider` replaces something already shorter
and clearer than its own name.

Reuse is one reason to make a component. Readability is the other, and it is the more
common one.